In [1]:
import os,sys
os.chdir('../')
%pwd

'/Users/hh/UOB/FYP/arabic-sentiment-framework'

In [2]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class DataPreprocessingConfig:
    root_dir: Path
    slsa_data_local_path: Path
    absa_data_train_local_path: Path
    absa_data_test_local_path: Path
    absa_data_test_gold_local_path: Path
    save_train_full_slsa_data: Path
    save_train_full_absa_data: Path
    save_train_small_slsa_data: Path
    save_train_small_absa_data: Path
    save_dev_slsa_data: Path
    save_dev_absa_data: Path
    save_test_slsa_data: Path
    save_test_absa_data: Path
    slsa_label_column: str
    slsa_text_column: str
    slsa_seed: float
    slsa_test_ratio: float
    slsa_dev_ratio: float
    slsa_small_n: float


    

In [3]:
from src.arabic_sentiment.constants import CONFIG_PATH,PARAMS_PATH,SCHEMA_PATH
from src.arabic_sentiment.utils.common import create_directories, read_yaml
class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(Path(CONFIG_PATH))
        self.params = read_yaml(Path(PARAMS_PATH))
        self.schema = read_yaml(Path(SCHEMA_PATH))
        create_directories([self.config.artifacts_root])
    
    def get_data_preprocessing_config(self) -> DataPreprocessingConfig:
        config = self.config.data_preprocessing
        params = self.params.data_preprocessing
        schema = self.schema
        data_preprocessing_config = DataPreprocessingConfig(
            root_dir= config.root_dir,
            slsa_data_local_path= config.slsa_data_local_path,
            absa_data_train_local_path= config.absa_data_train_local_path,
            absa_data_test_local_path= config.absa_data_test_local_path,
            absa_data_test_gold_local_path= config.absa_data_test_gold_local_path,
            save_train_full_slsa_data= config.save_train_full_slsa_data,
            save_train_full_absa_data= config.save_train_full_absa_data,
            save_train_small_slsa_data= config.save_train_small_slsa_data,
            save_train_small_absa_data= config.save_train_small_absa_data,
            save_dev_slsa_data= config.save_dev_slsa_data,
            save_dev_absa_data= config.save_dev_absa_data,
            save_test_slsa_data= config.save_test_slsa_data,
            save_test_absa_data= config.save_test_absa_data,
            slsa_label_column= schema.slsa.target_column.name,
            slsa_text_column= schema.slsa.columns.text.name,
            slsa_seed= params.slsa.seed,
            slsa_test_ratio= params.slsa.test_ratio,
            slsa_dev_ratio= params.slsa.dev_ratio,
            slsa_small_n= params.slsa.small_n
        )
        return data_preprocessing_config


In [4]:
from pathlib import Path
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from src.arabic_sentiment.logging import logger

In [9]:
class DataPreprocessing:
    def __init__(self, data_preprocessing_config:DataPreprocessingConfig):
        self.config = data_preprocessing_config
        create_directories([self.config.root_dir])

    def slsa_dataset_transform(self, source:Path):
        try:
            logger.info("Initiated slsa dataset transformation")
            source_slsa = self.config.slsa_data_local_path
            df = pd.read_csv(source_slsa, sep="\t", dtype=str)
            shuffled_df = df.sample(n=len(df))
            shuffled_df = shuffled_df[0:30001]
            logger.info("slsa dataset transformation completed")
            return shuffled_df
        except Exception as e:
            raise(e)

    def slsa_dataset_split(self, df:pd.DataFrame):
        try:
            logger.info("Initiated slsa dataset split")
            LABEL_COL = self.config.slsa_label_column
            SEED = self.config.slsa_seed
            TEST_RATIO = self.config.slsa_test_ratio
            DEV_RATIO = self.config.slsa_dev_ratio

            train_big_df, test_df = train_test_split(
                df,
                test_size=TEST_RATIO,
                random_state=SEED,
                stratify=df[LABEL_COL]
            )

            dev_size_relative_to_train = DEV_RATIO / (1 - TEST_RATIO)
            train_big_df, dev_df = train_test_split(
                train_big_df,
                test_size=dev_size_relative_to_train,
                random_state=SEED,
                stratify=train_big_df[LABEL_COL]
            )

            train_small_df = train_big_df.sample(n=self.config.slsa_small_n, random_state=SEED, replace=False)
            logger.info("slsa dataset splitting completed")
            return train_big_df, train_small_df, test_df, dev_df
        except Exception as e:
            raise(e)

    def slsa_df_to_jsonl(self,df, text_col, label_col, out_path):
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)  # <-- create folders
        try:
            with open(out_path, "w", encoding="utf-8") as f:
                for _, row in df.iterrows():
                    obj = {"text": row[text_col], "label": row[label_col]}
                    f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            logger.info(f"slsa DataFrame has been saved at {out_path}")
        except Exception as e:
            raise(e)
    
    def slsa_dataset_preprocessing(self):
        try:
            logger.info(f"slsa dataset preprocessing initiated")
            source_slsa = self.config.slsa_data_local_path
            df = self.slsa_dataset_transform(source_slsa)
            train_big_df, train_small_df, test_df, dev_df = self.slsa_dataset_split(df)
            LABEL_COL = self.config.slsa_label_column
            TEXT_COL = self.config.slsa_text_column
            self.slsa_df_to_jsonl(train_small_df, text_col=TEXT_COL, label_col=LABEL_COL, out_path=self.config.save_train_small_slsa_data)
            self.slsa_df_to_jsonl(train_big_df, text_col=TEXT_COL, label_col=LABEL_COL, out_path=self.config.save_train_full_slsa_data)
            self.slsa_df_to_jsonl(dev_df, text_col=TEXT_COL, label_col=LABEL_COL, out_path=self.config.save_dev_slsa_data)
            self.slsa_df_to_jsonl(test_df, text_col=TEXT_COL, label_col=LABEL_COL, out_path=self.config.save_test_slsa_data)
            logger.info(f"slsa dataset processing completed")
        except Exception as e:
            raise(e)

In [10]:
a = ConfigurationManager()
config = a.get_data_preprocessing_config()
obj = DataPreprocessing(config)
obj.slsa_dataset_preprocessing()



[2025-12-22 22:49:43,834] 13 src.arabic_sentiment.logging - INFO - entered read_yaml function at /Users/hh/UOB/FYP/arabic-sentiment-framework
[2025-12-22 22:49:43,838] 16 src.arabic_sentiment.logging - INFO - yaml file: config/config.yaml loaded
[2025-12-22 22:49:43,840] 13 src.arabic_sentiment.logging - INFO - entered read_yaml function at /Users/hh/UOB/FYP/arabic-sentiment-framework
[2025-12-22 22:49:43,841] 16 src.arabic_sentiment.logging - INFO - yaml file: params.yaml loaded
[2025-12-22 22:49:43,842] 13 src.arabic_sentiment.logging - INFO - entered read_yaml function at /Users/hh/UOB/FYP/arabic-sentiment-framework
[2025-12-22 22:49:43,844] 16 src.arabic_sentiment.logging - INFO - yaml file: schema.yaml loaded
[2025-12-22 22:49:43,845] 29 src.arabic_sentiment.logging - INFO - created directory at artifacts
[2025-12-22 22:49:43,846] 29 src.arabic_sentiment.logging - INFO - created directory at artifacts/data/processed
[2025-12-22 22:49:43,847] 61 src.arabic_sentiment.logging - INFO 